# 3.2 — Абляции, метрики по группам и out-of-distribution тесты

**Папка 3, подноутбук 2.** Анализ устойчивости и вклада компонентов: метрики по типам
грунта и режимам нагружения, абляции DPI-Flow и EVT-NeuralSSM, OOD-тесты (экстраполяция
«короткий → длинный горизонт» и удержанный регион слабых грунтов). Все рисунки и таблицы
— на английском.

## Окружение, данные и модели

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow,
                                    NeuralODENoPhysics, FlowNoODE)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from liquefaction_ai.evaluation import (filter_split, grouped_metrics, is_holdout_region,
                                        run_quick_experiment, subsample_split)
from liquefaction_ai.constants import LOAD_DISPLAY_NAMES_EN, SOIL_DISPLAY_NAMES_EN
from liquefaction_ai.viz import bar, grouped_bar

## Metrics by soil type and loading regime (DPI-Flow)

In [2]:
dpi, hp, _ = load_trained("dpi_flow")
out = collect_outputs(dpi, test, config, device)
_, sample_df = compute_metrics("DPI-Flow", out, test, config)
sample_df["soil_en"] = sample_df["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
sample_df["load_en"] = sample_df["load_mode"].map(LOAD_DISPLAY_NAMES_EN)
soil_metrics = grouped_metrics(sample_df, "soil_en")
load_metrics = grouped_metrics(sample_df, "load_en")
display(english_metric_table(soil_metrics).round(4))
grouped_bar(soil_metrics["soil_en"].tolist(),
            {"AUROC": soil_metrics["AUROC"].tolist(), "Trajectory RMSE": soil_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by soil type", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_soil").show()
grouped_bar(load_metrics["load_en"].tolist(),
            {"AUROC": load_metrics["AUROC"].tolist(), "Trajectory RMSE": load_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by loading regime", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_load").show()

,soil_en,N samples,Liquefaction rate,Mean predicted risk,Mean |ΔN_liq| (cycles),Mean log-error N_liq,Mean trajectory RMSE,Mean interval width,Physics violations,AUROC
3,Loam,23,0.5217,0.5816,2476.586426,2.3926,0.1394,0.5641,0.1304,0.9167
0,Clay,17,0.4706,0.5720,3301.677734,2.6232,0.0964,0.5356,0.1765,0.8889
2,Fine sand,17,0.4706,0.6689,2649.047852,2.8908,0.1474,0.5675,0.4118,0.7361
4,Medium sand,16,0.6875,0.6806,1620.550537,2.4094,0.1479,0.5672,0.1250,0.8727
1,Coarse sand,14,0.9286,0.6477,664.098206,1.3298,0.2912,0.6913,0.1429,0.3077
6,Silty sand,8,0.8750,0.7228,753.589600,2.0533,0.1758,0.5884,0.1250,1.0000
5,Sandy loam,6,0.5000,0.6269,3326.414795,2.8637,0.1092,0.5456,0.1667,0.8889


[save_figure] PNG для '3_2_metrics_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_2_metrics_by_load' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Component ablations

In [3]:
sub = min(config.ablation_subset, 2400)
atrain = subsample_split(benchmark["train"], int(sub * 0.7), config.seed)
aval = subsample_split(benchmark["val"], int(sub * 0.15), config.seed + 1)
atest = subsample_split(benchmark["test"], int(sub * 0.15), config.seed + 2)
sd, pdim, qd = static_dim, prefix_dim, seq_dim = (benchmark["train"]["static"].shape[1],
    benchmark["train"]["prefix_summary"].shape[1], benchmark["train"]["seq_in"].shape[-1])
mcr, sl, pl = config.max_cycle_reference, config.seq_len, config.prefix_len
ablation_specs = [
    ("DPI-Flow (full)", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2, use_analytical_layer=True)),
    ("DPI-Flow w/o calibration", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=True)),
    ("DPI-Flow w/o probabilistic head", DPIFlow(sd, pdim, sl, pl, mcr, probabilistic=False, calibration_steps=2)),
    ("DPI-Flow w/o ODE layer", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=False)),
    # --- покомпонентные абляции (изоляция вклада ODE / flow / физики) ---
    ("ODE w/o flow", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2, use_analytical_layer=True, use_flow=False)),
    ("Flow w/o ODE", FlowNoODE(sd, pdim, sl)),
    ("Neural ODE w/o physics", NeuralODENoPhysics(sd, qd)),
    ("EVT (full)", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)),
    ("EVT w/o trigger", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_trigger_head=False)),
    ("EVT w/o post-event dynamics", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, structured_post_event=False)),
    ("EVT w/o CRR damage", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_crr_damage=False)),
]
ablation_rows = [run_quick_experiment(n, m, atrain, aval, atest, epochs=config.ablation_epochs,
                                      config=config, device=device) for n, m in ablation_specs]
ablation_df = pd.DataFrame(ablation_rows).sort_values("Traj_RMSE")
display(english_metric_table(ablation_df)[["Model", "Trajectory RMSE", "AUROC", "Brier"]].round(4))
bar(ablation_df["model"], ablation_df["Traj_RMSE"], title="Ablations: trajectory RMSE (lower is better)",
    ylabel="Trajectory RMSE (–)", color="#6610f2", horizontal=True,
    save=SAVE_FIGS, fig_id="3_2_ablations_rmse").show()

[DPI-Flow (full)] эпоха 01 | обучение=2.5298 | валидация=-0.0611


[DPI-Flow (full)] эпоха 02 | обучение=-0.3949 | валидация=-0.7578


[DPI-Flow w/o calibration] эпоха 01 | обучение=3.5219 | валидация=-0.1063


[DPI-Flow w/o calibration] эпоха 02 | обучение=-0.4001 | валидация=-0.7509


[DPI-Flow w/o probabilistic head] эпоха 01 | обучение=2.5924 | валидация=0.1924


...

[EVT w/o post-event dynamics] эпоха 02 | обучение=1.4162 | валидация=1.3201


[EVT w/o CRR damage] эпоха 01 | обучение=3.9681 | валидация=1.6182


[EVT w/o CRR damage] эпоха 02 | обучение=0.5442 | валидация=-0.2504


,Model,Trajectory RMSE,AUROC,Brier
1,DPI-Flow w/o calibration,0.1729,0.6543,0.2560
0,DPI-Flow (full),0.1729,0.6911,0.2541
2,DPI-Flow w/o probabilistic head,0.1847,0.6551,0.2413
7,EVT (full),0.1852,0.3916,0.3407
4,ODE w/o flow,0.1948,0.2146,0.3103
10,EVT w/o CRR damage,0.2129,0.0054,0.3485
3,DPI-Flow w/o ODE layer,0.2282,1.0000,0.0198
6,Neural ODE w/o physics,0.2372,0.9955,0.2017
8,EVT w/o trigger,0.2390,0.6501,0.2856
5,Flow w/o ODE,0.2520,0.9983,0.0347


[save_figure] PNG для '3_2_ablations_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Out-of-distribution tests

In [4]:
train_meta = benchmark["train"]["meta"]; val_meta = benchmark["val"]["meta"]; test_meta = benchmark["test"]["meta"]
short_thr = float(train_meta["N_max"].quantile(0.60)); long_thr = float(test_meta["N_max"].quantile(0.80))
short_train = filter_split(benchmark["train"], train_meta["N_max"].to_numpy() <= short_thr)
short_val = filter_split(benchmark["val"], val_meta["N_max"].to_numpy() <= short_thr)
long_test = filter_split(benchmark["test"], test_meta["N_max"].to_numpy() >= long_thr)
e_thr = float(benchmark["meta"]["e"].quantile(0.75)); vs_thr = float(benchmark["meta"]["V_s"].quantile(0.25))
region_test = filter_split(benchmark["test"], is_holdout_region(test_meta, e_thr, vs_thr))
hold_train = filter_split(benchmark["train"], ~is_holdout_region(train_meta, e_thr, vs_thr))
hold_val = filter_split(benchmark["val"], ~is_holdout_region(val_meta, e_thr, vs_thr))
short_train = subsample_split(short_train, 1100, config.seed); short_val = subsample_split(short_val, 300, config.seed + 1)
long_test = subsample_split(long_test, 700, config.seed + 2)
hold_train = subsample_split(hold_train, 1100, config.seed + 3); hold_val = subsample_split(hold_val, 300, config.seed + 4)
region_test = subsample_split(region_test, 700, config.seed + 5)

def fresh(kind):
    if kind == "tcn": return TCNBaseline(sd, qd, 96)
    if kind == "dpi": return DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2)
    return EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)

ood_rows = []
for disp, kind in [("TCN", "tcn"), ("DPI-Flow", "dpi"), ("EVT-NeuralSSM", "evt")]:
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), short_train, short_val, long_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "short→long"})
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), hold_train, hold_val, region_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "held-out region"})
ood_df = pd.DataFrame(ood_rows)
display(english_metric_table(ood_df)[["Model", "test", "Trajectory RMSE", "AUROC"]].round(4))
sl_df = ood_df[ood_df["test"] == "short→long"]; rg_df = ood_df[ood_df["test"] == "held-out region"]
grouped_bar(sl_df["model"].tolist(),
            {"short → long horizon": sl_df["Traj_RMSE"].tolist(), "held-out region": rg_df["Traj_RMSE"].tolist()},
            title="Out-of-distribution trajectory RMSE (lower is better)", ylabel="Trajectory RMSE (–)",
            save=SAVE_FIGS, fig_id="3_2_ood_rmse").show()

[TCN] эпоха 01 | обучение=0.2721 | валидация=0.2041


[TCN] эпоха 02 | обучение=0.1636 | валидация=-0.1211


[TCN] эпоха 01 | обучение=0.2840 | валидация=0.2232


[TCN] эпоха 02 | обучение=0.1941 | валидация=-0.0555


[DPI-Flow] эпоха 01 | обучение=3.8057 | валидация=0.2521


...

[EVT-NeuralSSM] эпоха 02 | обучение=0.1803 | валидация=-0.6763


[EVT-NeuralSSM] эпоха 01 | обучение=3.5753 | валидация=0.0606


[EVT-NeuralSSM] эпоха 02 | обучение=-0.3234 | валидация=-0.2880


,Model,test,Trajectory RMSE,AUROC
0,TCN,short→long,0.2293,0.3846
1,TCN,held-out region,0.3602,1.0000
2,DPI-Flow,short→long,0.1934,0.7692
3,DPI-Flow,held-out region,0.1896,0.1667
4,EVT-NeuralSSM,short→long,0.1605,0.9487
5,EVT-NeuralSSM,held-out region,0.2196,0.4444


[save_figure] PNG для '3_2_ood_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Абляции подтверждают вклад компонентов; структурированные модели устойчивее в OOD.
Дальше — **3.3 разбор кейсов**.

In [5]:
# === OOD-разбиение: качество по площадкам, типам грунтов и диапазонам CSR ===
# Для основной модели (DPI-Flow) считаем метрики по группам — генерализация по доменам.
from liquefaction_ai.evaluation import grouped_metrics
dpi_full, _, _ = load_trained("dpi_flow")
_out = collect_outputs(dpi_full, test, config, device)
_, sdf = compute_metrics("DPI-Flow", _out, test, config)
sdf["soil_en"] = sdf["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
import numpy as np
sdf["CSR_bin"] = pd.cut(sdf["CSR_base"], [0, 0.12, 0.18, 0.25, 1.0],
                         labels=["CSR≤0.12", "0.12–0.18", "0.18–0.25", "CSR>0.25"])
_show = ["samples", "mean_traj_rmse", "mean_nliq_log_err", "physics_violation_rate", "AUROC"]
for dim, label in [("object", "площадка (site)"), ("soil_en", "тип грунта"), ("CSR_bin", "диапазон CSR")]:
    g = grouped_metrics(sdf, dim)
    print(f"\n=== OOD по: {label} ===")
    display(english_metric_table(g[[dim] + _show]).round(4))
    bar(g[dim].astype(str), g["mean_traj_rmse"], title=f"DPI-Flow: PPR RMSE by {dim}",
        ylabel="Trajectory RMSE (–)", color="#0b6efd", horizontal=True,
        save=SAVE_FIGS, fig_id=f"3_2_ood_{dim}").show()


=== OOD по: площадка (site) ===


,object,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
10,Штормовое разжижение/856-23 3-й Хорошевский ...,21,0.1405,2.1912,0.1905,0.8056
2,Потенциал разжижения/852-23 3-й Хорошевский ...,17,0.1916,1.9647,0.0000,NaN
0,Потенциал разжижения/333-24 Дубнинская - Plaxi...,13,0.1505,2.1352,0.0000,NaN
1,Потенциал разжижения/338-24 АД Москва - Казань...,12,0.1633,2.2545,0.0000,NaN
3,Штормовое разжижение/196-24 ул. Б. Татарская,10,0.1708,2.5345,0.1000,0.9167
9,Штормовое разжижение/818-23 Руза гора - plaxis...,8,0.1102,3.2888,0.3750,0.7500
5,Штормовое разжижение/255-24 Б. Садовая 8 - pla...,6,0.1170,3.8304,1.0000,NaN
6,Штормовое разжижение/390-24 Шаболовская -Plaxi...,6,0.0689,2.2586,0.3333,NaN
8,Штормовое разжижение/638-24 Ленинградское шоссе,5,0.3292,1.1893,0.0000,1.0000
7,Штормовое разжижение/613-23 Тушино - plaxis (23),2,0.1217,3.6241,1.0000,NaN


[save_figure] PNG для '3_2_ood_object' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




=== OOD по: тип грунта ===


,soil_en,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
3,Loam,23,0.1394,2.3926,0.1304,0.9167
0,Clay,17,0.0964,2.6232,0.1765,0.8889
2,Fine sand,17,0.1474,2.8908,0.4118,0.7361
4,Medium sand,16,0.1479,2.4094,0.1250,0.8727
1,Coarse sand,14,0.2912,1.3298,0.1429,0.3077
6,Silty sand,8,0.1758,2.0533,0.1250,1.0000
5,Sandy loam,6,0.1092,2.8637,0.1667,0.8889


[save_figure] PNG для '3_2_ood_soil_en' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido




=== OOD по: диапазон CSR ===


/sessions/determined-cool-fermat/mnt/liquefaction-ai/src/liquefaction_ai/evaluation/metrics.py:527: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample_df.groupby(group_col)


,CSR_bin,N samples,Mean trajectory RMSE,Mean log-error N_liq,Physics violations,AUROC
1,0.12–0.18,39,0.1463,2.5955,0.1026,0.7929
0,CSR≤0.12,37,0.0897,2.9927,0.4054,1.0000
3,CSR>0.25,14,0.4140,0.5187,0.0000,NaN
2,0.18–0.25,11,0.0942,1.8479,0.0000,NaN


[save_figure] PNG для '3_2_ood_CSR_bin' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

